# Introduction

This notebook aims to generate a synthetic test dataset to fully test the functionality of the API. The provided test.parquet contains only one batch of example data, which is not enough to cover all coner cases in the submission. By generating a synthetic test data using the train.parquet, we can better simulate the submission and identify potential bugs in the code. This is especially crucial for the time-series competition, as our code will go through a 3-month online testing without a chance to debug. 

**<span style="color:red">- WARNING: DO NOT FORGET TO CHANGE THE SYNTHETIC TEST BACK TO THE REAL TEST BEFORE SUBMISSION !!! -</span>**

In [1]:
import os
import polars as pl
import pandas as pd 
from pathlib import Path
from tqdm import tqdm

import kaggle_evaluation.jane_street_inference_server as js_server

DATA_DIR = Path('/kaggle/input/jane-street-real-time-market-data-forecasting')

In [2]:
date_offset = 1690

pl_all = pl.scan_parquet(DATA_DIR/"train.parquet").filter(pl.col("date_id") >= date_offset-1).collect()

# Make synthetic test dataset

In [3]:
# make syn_test 
syn_test = pl_all.with_columns(
    pl.lit(True).alias("is_scored"),
    pl.col('date_id') - date_offset
    ).with_row_index(name="row_id", offset=0)

syn_test = syn_test.select(
    ['row_id', 'date_id', 'time_id', 'symbol_id', 'weight', 'is_scored'] + [f'feature_{x:02}' for x in range(79)]
)

syn_test_partition = syn_test.partition_by('date_id', maintain_order=True, as_dict=True)

output_dir = "synthetic_test.parquet"
os.makedirs(output_dir, exist_ok=True)

row_id_offset = syn_test.filter(pl.col('date_id')<0).select('row_id').max().item()
print("row_id_offset:", row_id_offset)

for key, _df in syn_test_partition.items():
    if key[0] >= 0:
        os.makedirs(f"{output_dir}/date_id={key[0]}", exist_ok=True)
        _df = _df.with_columns(pl.col('row_id')-row_id_offset)
        _df.write_parquet(f"{output_dir}/date_id={key[0]}/part-0.parquet")

row_id_offset: 37751


In [4]:
syn_test

row_id,date_id,time_id,symbol_id,weight,is_scored,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,…,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78
u32,i16,i16,i8,f32,bool,f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,-1,0,0,3.920472,true,2.838797,1.213867,3.447698,3.246968,3.127066,-1.223813,-0.103129,-0.365642,0.369916,11,7,76,-0.704927,3.833137,0.574594,null,-0.399141,null,-0.984176,-1.927787,0.725419,-0.071851,1.639402,0.531733,1.802795,1.705573,1.106273,1.002087,0.830674,-0.913411,-0.763544,…,null,-0.948574,null,-1.029105,1.431838,-1.532047,-1.185343,-1.695741,null,1.584205,null,null,-0.817087,null,-0.841178,1.402767,null,-0.395893,-0.486984,0.203422,-0.305938,-0.328256,-0.336665,-1.867303,-3.241649,-0.654473,2.82454,0.233303,-0.668989,2.801647,0.310046,null,null,1.031332,0.868393,0.128927,0.020791
1,-1,0,1,3.277961,true,2.63848,1.530045,3.172238,3.135972,3.163308,-1.022497,-0.118519,-0.463337,0.348295,11,7,76,-0.281585,4.929662,0.652993,null,-0.289926,null,-1.200994,-1.991386,0.693848,0.044299,1.134433,0.536771,1.342152,1.577414,-1.357036,-0.264299,1.068754,-0.604443,-0.756437,…,null,0.350526,null,-0.551633,1.932005,-2.003859,1.563956,-0.533985,null,2.449599,null,null,-0.412837,null,0.202788,1.607015,null,5.021396,0.854807,0.203422,-0.223912,-0.260981,-0.306294,-1.287731,-1.372433,-0.495636,6.469605,0.878077,-0.573693,4.97798,0.698142,null,null,1.12372,0.494466,-0.020493,-0.102544
2,-1,0,2,2.905656,true,3.173025,0.733732,3.086998,3.173442,3.085099,-1.239584,-0.137361,-0.660561,0.48729,81,2,59,-0.990263,0.554075,-0.588245,null,-0.603307,null,-2.008288,-1.387164,0.072834,-0.155172,0.871501,0.812759,0.481315,0.02082,0.617991,0.398821,-0.055455,-0.614166,-0.731085,…,null,0.564412,null,-1.37274,2.116828,0.620467,0.503527,1.166342,null,0.685441,null,null,-0.216445,null,-2.186445,2.357004,null,0.188205,-0.06112,0.203422,-0.428063,-0.225762,-0.573654,-1.511528,-1.587279,-1.011688,0.502076,-0.363863,-1.041157,0.300714,-0.492709,null,null,-0.059283,-0.042962,-0.170971,-0.203765
3,-1,0,3,1.902979,true,2.906444,1.637979,3.418966,2.833408,3.377959,-1.226049,-0.157936,-0.777115,0.598883,4,3,11,-0.613559,4.477407,0.11501,null,-0.505266,null,-0.917451,-1.596013,-0.041529,-0.010549,0.767809,0.417076,0.37292,-0.053514,-0.015659,-0.45608,-0.609566,-0.497994,-0.611594,…,null,0.180711,null,-1.324028,1.448735,-0.387757,0.863922,0.282344,null,1.347153,null,null,1.020928,null,-1.978639,1.979012,null,2.028311,1.088718,0.203422,-0.350207,-0.403734,-0.340905,-1.236425,-1.805918,-0.660882,3.443619,0.240645,-0.767834,2.457366,-0.117942,null,null,0.561677,0.5911,-0.03369,-0.079931
4,-1,0,4,2.496076,true,2.914232,1.244243,2.860873,2.807699,2.353768,-0.625057,-0.079814,-0.338505,0.19907,15,1,9,-0.876844,0.348778,-0.570208,null,-0.707293,null,-0.940803,-1.462916,-1.101414,1.050136,1.163255,0.268036,4.123104,2.223214,-1.272668,-1.040628,-0.920485,-0.66073,-0.87472,…,null,0.001017,null,-0.948041,1.418141,0.482694,0.527924,0.452903,null,-0.038147,null,null,-0.646144,null,-2.362279,1.921825,null,-0.46985,-0.264322,0.203422,-0.400293,-0.236181,-0.316313,-1.519481,-1.678275,-0.610796,0.08896,-0.6

In [5]:
pl_test = pl.read_parquet(DATA_DIR/"test.parquet", n_rows=10000)
pl_test

row_id,date_id,time_id,symbol_id,weight,is_scored,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,…,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78
i64,i16,i16,i8,f32,bool,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,0,0,0,3.169998,true,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,null,-0.0,null,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,0.0,-0.0,0.0,0.0,null,0.0,null,null,-0.0,null,-0.0,0.0,null,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,null,null,0.0,0.0,-0.0,-0.0
1,0,0,1,2.165993,true,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,null,-0.0,null,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,-0.0,null,-0.0,0.0,null,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,null,null,0.0,0.0,0.0,0.0
2,0,0,2,3.06555,true,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,null,-0.0,null,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,0.0,-0.0,-0.0,-0.0,null,0.0,null,null,-0.0,null,-0.0,0.0,null,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,null,null,0.0,0.0,-0.0,-0.0
3,0,0,3,2.698642,true,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,null,-0.0,null,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,0.0,-0.0,0.0,-0.0,null,-0.0,null,null,-0.0,null,-0.0,0.0,null,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,null,null,0.0,0.0,-0.0,-0.0
4,0,0,4,1.80333,true,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,null,-0.0,null,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,-0.0,null,-0.0,0.0,null,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,null,null,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
34,0,0,34,3.240565,true,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,null,-0.0,null,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,-0.0,-0.0,-0.0,-0.0,null,-0.0,null,null,-0.0,null,-0.0,-0.0,null,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0
35,0,0,35,1.057221,true,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,null,-0.0,null,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,…,null,0.0,null,-0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,-0.0,0.0,null,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,null,null,0.0,0.0,-0.0,-0.0
36,0,0,36,0.907022,true,0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,null,-0.0,null,-0.0,-0.0,-0.0,null,-0.0,-0.0,0.0,-0.0,null,null,-0.0,-0.0,-0.0,…,null,-0.0,null,-0.0,0.0,-0.0,0.0,0.0,null,0.0,null,null,0.0,null,-0.0,0.0,null,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,null,null,0.0,0.0,0.0,0.0


# Make synthetic lags

In [6]:
# make syn_lag

syn_lag = pl_all.select(
    ['date_id', 'time_id', 'symbol_id'] + [f'responder_{x}' for x in range(9)]
).with_columns(pl.col('date_id')-date_offset)

syn_lag = syn_lag.rename({f'responder_{x}': f'responder_{x}_lag_1' for x in range(9)})

syn_lag_partition = syn_lag.partition_by('date_id', maintain_order=True, as_dict=True)

output_dir = "synthetic_lag.parquet"
os.makedirs(output_dir, exist_ok=True)

for key, _df in syn_lag_partition.items():
    os.makedirs(f"{output_dir}/date_id={key[0]+1}", exist_ok=True)
    _df = _df.with_columns(pl.col('date_id')+1)
    _df.write_parquet(f"{output_dir}/date_id={key[0]+1}/part-0.parquet")

In [7]:
pl_lag = pl.read_parquet(DATA_DIR / 'lags.parquet')
pl_lag

date_id,time_id,symbol_id,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
i16,i16,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,0,0,-0.442215,-0.322407,0.143594,-0.92689,-0.782236,-0.036595,-1.305746,-0.795677,-0.143724
0,0,1,-0.651829,-1.70784,-0.893942,-1.065488,-1.871338,-0.615652,-1.162801,-1.205924,-1.245934
0,0,2,-0.656373,-0.264575,-0.892879,-1.511886,-1.03348,-0.378265,-1.57429,-1.863071,-0.027343
0,0,3,-0.188186,-0.19097,-0.70149,0.098453,-1.015506,-0.054984,0.329152,-0.965471,0.576635
0,0,4,-0.257462,-0.471325,-0.29742,0.074018,-0.324194,-0.597093,0.219856,-0.276356,-0.90479
…,…,…,…,…,…,…,…,…,…,…,…
0,0,34,-0.185392,-0.187891,-0.206658,-0.634903,-0.643175,-0.443875,-0.556474,-1.122211,-0.884185
0,0,35,-0.308923,-0.434147,-1.354941,0.30054,-0.830827,0.424937,0.518839,-0.687369,1.440577
0,0,36,-0.074661,-0.261698,-0.007051,-2.60039,-1.146709,-1.601274,-3.216254,-1.249338,-2.868875


# Test submission using the synthetic test & lags

In [8]:
from collections import defaultdict
import numpy as np

def get_data_from_batch(batch):
    ts_window = batch[-100:]
    x = ts_window[[f'feature_{x:02}' for x in range(79)]].to_numpy()
    w = ts_window['weight'].to_numpy() 
    
    return x, w

# def model(x):
#     return np.nanmean(x, axis=(-1,-2))

In [9]:
!pip install /kaggle/input/rtdl-num/rtdl_num_embeddings-0.0.11-py3-none-any.whl

Processing /kaggle/input/rtdl-num/rtdl_num_embeddings-0.0.11-py3-none-any.whl


In [10]:
%%time
# License: https://github.com/yandex-research/tabm/blob/main/LICENSE

# NOTE
# The minimum required versions of the dependencies are specified in README.md.

import itertools
from typing import Any, Literal

import rtdl_num_embeddings
import torch
import torch.nn as nn
from torch import Tensor


# ======================================================================================
# Initialization
# ======================================================================================
def init_rsqrt_uniform_(x: Tensor, d: int) -> Tensor:
    assert d > 0
    d_rsqrt = d**-0.5
    return nn.init.uniform_(x, -d_rsqrt, d_rsqrt)


@torch.inference_mode()
def init_random_signs_(x: Tensor) -> Tensor:
    return x.bernoulli_(0.5).mul_(2).add_(-1)


# ======================================================================================
# Modules
# ======================================================================================
class NLinear(nn.Module):
    """N linear layers applied in parallel to N disjoint parts of the input.

    **Shape**

    - Input: ``(B, N, in_features)``
    - Output: ``(B, N, out_features)``

    The i-th linear layer is applied to the i-th matrix of the shape (B, in_features).

    Technically, this is a simplified version of delu.nn.NLinear:
    https://yura52.github.io/delu/stable/api/generated/delu.nn.NLinear.html.
    The difference is that this layer supports only 3D inputs
    with exactly one batch dimension. By contrast, delu.nn.NLinear supports
    any number of batch dimensions.
    """

    def __init__(
        self, n: int, in_features: int, out_features: int, bias: bool = True
    ) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n, in_features, out_features))
        self.bias = nn.Parameter(torch.empty(n, out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        d = self.weight.shape[-2]
        init_rsqrt_uniform_(self.weight, d)
        if self.bias is not None:
            init_rsqrt_uniform_(self.bias, d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.ndim == 3
        assert x.shape[-(self.weight.ndim - 1) :] == self.weight.shape[:-1]

        x = x.transpose(0, 1)
        x = x @ self.weight
        x = x.transpose(0, 1)
        if self.bias is not None:
            x = x + self.bias
        return x


class OneHotEncoding0d(nn.Module):
    # Input:  (*, n_cat_features=len(cardinalities))
    # Output: (*, sum(cardinalities))

    def __init__(self, cardinalities: list[int]) -> None:
        super().__init__()
        self._cardinalities = cardinalities

    def forward(self, x: Tensor) -> Tensor:
        assert x.ndim >= 1
        assert x.shape[-1] == len(self._cardinalities)

        return torch.cat(
            [
                # NOTE
                # This is a quick hack to support out-of-vocabulary categories.
                #
                # Recall that lib.data.transform_cat encodes categorical features
                # as follows:
                # - In-vocabulary values receive indices from `range(cardinality)`.
                # - All out-of-vocabulary values (i.e. new categories in validation
                #   and test data that are not presented in the training data)
                #   receive the index `cardinality`.
                #
                # As such, the line below will produce the standard one-hot encoding for
                # known categories, and the all-zeros encoding for unknown categories.
                # This may not be the best approach to deal with unknown values,
                # but should be enough for our purposes.
                nn.functional.one_hot(x[..., i], cardinality + 1)[..., :-1]
                for i, cardinality in enumerate(self._cardinalities)
            ],
            -1,
        )


class ScaleEnsemble(nn.Module):
    def __init__(
        self,
        k: int,
        d: int,
        *,
        init: Literal['ones', 'normal', 'random-signs'],
    ) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(k, d))
        self._weight_init = init
        self.reset_parameters()

    def reset_parameters(self) -> None:
        if self._weight_init == 'ones':
            nn.init.ones_(self.weight)
        elif self._weight_init == 'normal':
            nn.init.normal_(self.weight)
        elif self._weight_init == 'random-signs':
            init_random_signs_(self.weight)
        else:
            raise ValueError(f'Unknown weight_init: {self._weight_init}')

    def forward(self, x: Tensor) -> Tensor:
        assert x.ndim >= 2
        return x * self.weight


class LinearEfficientEnsemble(nn.Module):
    """
    This layer is a more configurable version of the "BatchEnsemble" layer
    from the paper
    "BatchEnsemble: An Alternative Approach to Efficient Ensemble and Lifelong Learning"
    (link: https://arxiv.org/abs/2002.06715).

    First, this layer allows to select only some of the "ensembled" parts:
    - the input scaling  (r_i in the BatchEnsemble paper)
    - the output scaling (s_i in the BatchEnsemble paper)
    - the output bias    (not mentioned in the BatchEnsemble paper,
                          but is presented in public implementations)

    Second, the initialization of the scaling weights is configurable
    through the `scaling_init` argument.

    NOTE
    The term "adapter" is used in the TabM paper only to tell the story.
    The original BatchEnsemble paper does NOT use this term. So this class also
    avoids the term "adapter".
    """

    r: None | Tensor
    s: None | Tensor
    bias: None | Tensor

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        *,
        k: int,
        ensemble_scaling_in: bool,
        ensemble_scaling_out: bool,
        ensemble_bias: bool,
        scaling_init: Literal['ones', 'random-signs'],
    ):
        assert k > 0
        if ensemble_bias:
            assert bias
        super().__init__()

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.register_parameter(
            'r',
            (
                nn.Parameter(torch.empty(k, in_features))
                if ensemble_scaling_in
                else None
            ),  # type: ignore[code]
        )
        self.register_parameter(
            's',
            (
                nn.Parameter(torch.empty(k, out_features))
                if ensemble_scaling_out
                else None
            ),  # type: ignore[code]
        )
        self.register_parameter(
            'bias',
            (
                nn.Parameter(torch.empty(out_features))  # type: ignore[code]
                if bias and not ensemble_bias
                else nn.Parameter(torch.empty(k, out_features))
                if ensemble_bias
                else None
            ),
        )

        self.in_features = in_features
        self.out_features = out_features
        self.k = k
        self.scaling_init = scaling_init

        self.reset_parameters()

    def reset_parameters(self):
        init_rsqrt_uniform_(self.weight, self.in_features)
        scaling_init_fn = {'ones': nn.init.ones_, 'random-signs': init_random_signs_}[
            self.scaling_init
        ]
        if self.r is not None:
            scaling_init_fn(self.r)
        if self.s is not None:
            scaling_init_fn(self.s)
        if self.bias is not None:
            bias_init = torch.empty(
                # NOTE: the shape of bias_init is (out_features,) not (k, out_features).
                # It means that all biases have the same initialization.
                # This is similar to having one shared bias plus
                # k zero-initialized non-shared biases.
                self.out_features,
                dtype=self.weight.dtype,
                device=self.weight.device,
            )
            bias_init = init_rsqrt_uniform_(bias_init, self.in_features)
            with torch.inference_mode():
                self.bias.copy_(bias_init)

    def forward(self, x: Tensor) -> Tensor:
        # x.shape == (B, K, D)
        assert x.ndim == 3

        # >>> The equation (5) from the BatchEnsemble paper (arXiv v2).
        if self.r is not None:
            x = x * self.r
        x = x @ self.weight.T
        if self.s is not None:
            x = x * self.s
        # <<<

        if self.bias is not None:
            x = x + self.bias
        return x


class MLP(nn.Module):
    def __init__(
        self,
        *,
        d_in: None | int = None,
        d_out: None | int = None,
        n_blocks: int,
        d_block: int,
        dropout: float,
        activation: str = 'ReLU',
    ) -> None:
        super().__init__()

        d_first = d_block if d_in is None else d_in
        self.blocks = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d_first if i == 0 else d_block, d_block),
                    getattr(nn, activation)(),
                    nn.Dropout(dropout),
                )
                for i in range(n_blocks)
            ]
        )
        self.output = None if d_out is None else nn.Linear(d_block, d_out)

    def forward(self, x: Tensor) -> Tensor:
        for block in self.blocks:
            x = block(x)
        if self.output is not None:
            x = self.output(x)
        return x


def make_efficient_ensemble(module: nn.Module, **kwargs) -> None:
    """Replace torch.nn.Linear modules with LinearEfficientEnsemble.

    NOTE
    In the paper, there are no experiments with networks with normalization layers.
    Perhaps, their trainable weights (the affine transformations) also need
    "ensemblification" as in the paper about "FiLM-Ensemble".
    Additional experiments are required to make conclusions.
    """
    for name, submodule in list(module.named_children()):
        if isinstance(submodule, nn.Linear):
            module.add_module(
                name,
                LinearEfficientEnsemble(
                    in_features=submodule.in_features,
                    out_features=submodule.out_features,
                    bias=submodule.bias is not None,
                    **kwargs,
                ),
            )
        else:
            make_efficient_ensemble(submodule, **kwargs)


def _get_first_ensemble_layer(backbone: MLP) -> LinearEfficientEnsemble:
    if isinstance(backbone, MLP):
        return backbone.blocks[0][0]  # type: ignore[code]
    else:
        raise RuntimeError(f'Unsupported backbone: {backbone}')


@torch.inference_mode()
def _init_first_adapter(
    weight: Tensor,
    distribution: Literal['normal', 'random-signs'],
    init_sections: list[int],
) -> None:
    """Initialize the first adapter.

    NOTE
    The `init_sections` argument is a historical artifact that accidentally leaked
    from irrelevant experiments to the final models. Perhaps, the code related
    to `init_sections` can be simply removed, but this was not tested.
    """
    assert weight.ndim == 2
    assert weight.shape[1] == sum(init_sections)

    if distribution == 'normal':
        init_fn_ = nn.init.normal_
    elif distribution == 'random-signs':
        init_fn_ = init_random_signs_
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    section_bounds = [0, *torch.tensor(init_sections).cumsum(0).tolist()]
    for i in range(len(init_sections)):
        # NOTE
        # As noted above, this section-based initialization is an arbitrary historical
        # artifact. Consider the first adapter of one ensemble member.
        # This adapter vector is implicitly split into "sections",
        # where one section corresponds to one feature. The code below ensures that
        # the adapter weights in one section are initialized with the same random value
        # from the given distribution.
        w = torch.empty((len(weight), 1), dtype=weight.dtype, device=weight.device)
        init_fn_(w)
        weight[:, section_bounds[i] : section_bounds[i + 1]] = w


_CUSTOM_MODULES = {
    # https://docs.python.org/3/library/stdtypes.html#definition.__name__
    CustomModule.__name__: CustomModule
    for CustomModule in [
        rtdl_num_embeddings.LinearEmbeddings,
        rtdl_num_embeddings.LinearReLUEmbeddings,
        rtdl_num_embeddings.PeriodicEmbeddings,
        rtdl_num_embeddings.PiecewiseLinearEmbeddings,
        MLP,
    ]
}


def make_module(type: str, *args, **kwargs) -> nn.Module:
    Module = getattr(nn, type, None)
    if Module is None:
        Module = _CUSTOM_MODULES[type]
    return Module(*args, **kwargs)


# ======================================================================================
# Optimization
# ======================================================================================
def default_zero_weight_decay_condition(
    module_name: str, module: nn.Module, parameter_name: str, parameter: nn.Parameter
):
    from rtdl_num_embeddings import _Periodic

    del module_name, parameter
    return parameter_name.endswith('bias') or isinstance(
        module,
        nn.BatchNorm1d
        | nn.LayerNorm
        | nn.InstanceNorm1d
        | rtdl_num_embeddings.LinearEmbeddings
        | rtdl_num_embeddings.LinearReLUEmbeddings
        | _Periodic,
    )


def make_parameter_groups(
    module: nn.Module,
    zero_weight_decay_condition=default_zero_weight_decay_condition,
    custom_groups: None | list[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    if custom_groups is None:
        custom_groups = []
    custom_params = frozenset(
        itertools.chain.from_iterable(group['params'] for group in custom_groups)
    )
    assert len(custom_params) == sum(
        len(group['params']) for group in custom_groups
    ), 'Parameters in custom_groups must not intersect'
    zero_wd_params = frozenset(
        p
        for mn, m in module.named_modules()
        for pn, p in m.named_parameters()
        if p not in custom_params and zero_weight_decay_condition(mn, m, pn, p)
    )
    default_group = {
        'params': [
            p
            for p in module.parameters()
            if p not in custom_params and p not in zero_wd_params
        ]
    }
    return [
        default_group,
        {'params': list(zero_wd_params), 'weight_decay': 0.0},
        *custom_groups,
    ]


# ======================================================================================
# The model
# ======================================================================================
class Model(nn.Module):
    """MLP & TabM."""

    def __init__(
        self,
        *,
        n_num_features: int,
        cat_cardinalities: list[int],
        n_classes: None | int,
        backbone: dict,
        bins: None | list[Tensor],  # For piecewise-linear encoding/embeddings.
        num_embeddings: None | dict = None,
        arch_type: Literal[
            # Plain feed-forward network without any kind of ensembling.
            'plain',
            #
            # TabM-mini
            'tabm-mini',
            #
            # TabM-mini. The first adapter is initialized from the normal distribution.
            # This is used in Section 5.1 of the paper.
            'tabm-mini-normal',
            #
            # TabM
            'tabm',
            #
            # TabM. The first adapter is initialized from the normal distribution.
            # This variation is not used in the paper, but there is a preliminary
            # evidence that may be a better default strategy.
            'tabm-normal',
        ],
        k: None | int = None,
    ) -> None:
        # >>> Validate arguments.
        assert n_num_features >= 0
        assert n_num_features or cat_cardinalities
        if arch_type == 'plain':
            assert k is None
        else:
            assert k is not None
            assert k > 0

        super().__init__()

        # >>> Continuous (numerical) features
        first_adapter_sections = []  # See the comment in `_init_first_adapter`.

        if n_num_features == 0:
            assert bins is None
            self.num_module = None
            d_num = 0

        elif num_embeddings is None:
            assert bins is None
            self.num_module = None
            d_num = n_num_features
            first_adapter_sections.extend(1 for _ in range(n_num_features))

        else:
            if bins is None:
                self.num_module = make_module(
                    **num_embeddings, n_features=n_num_features
                )
            else:
                assert num_embeddings['type'].startswith('PiecewiseLinearEmbeddings')
                self.num_module = make_module(**num_embeddings, bins=bins)
            d_num = n_num_features * num_embeddings['d_embedding']
            first_adapter_sections.extend(
                num_embeddings['d_embedding'] for _ in range(n_num_features)
            )

        # >>> Categorical features
        self.cat_module = (
            OneHotEncoding0d(cat_cardinalities) if cat_cardinalities else None
        )
        first_adapter_sections.extend(cat_cardinalities)
        d_cat = sum(cat_cardinalities)

        # >>> Backbone
        d_flat = d_num + d_cat
        self.minimal_ensemble_adapter = None
        # Any backbone can be here but we provide only MLP
        self.backbone = make_module(d_in=d_flat, **backbone)

        if arch_type != 'plain':
            assert k is not None
            first_adapter_init = (
                'normal'
                if arch_type in ('tabm-mini-normal', 'tabm-normal')
                # For other arch_types, the initialization depends
                # on the presense of num_embeddings.
                else 'random-signs'
                if num_embeddings is None
                else 'normal'
            )

            if arch_type in ('tabm-mini', 'tabm-mini-normal'):
                # Minimal ensemble
                self.minimal_ensemble_adapter = ScaleEnsemble(
                    k,
                    d_flat,
                    init='random-signs' if num_embeddings is None else 'normal',
                )
                _init_first_adapter(
                    self.minimal_ensemble_adapter.weight,  # type: ignore[code]
                    first_adapter_init,
                    first_adapter_sections,
                )

            elif arch_type in ('tabm', 'tabm-normal'):
                # Like BatchEnsemble, but all multiplicative adapters,
                # except for the very first one, are initialized with ones.
                make_efficient_ensemble(
                    self.backbone,
                    k=k,
                    ensemble_scaling_in=True,
                    ensemble_scaling_out=True,
                    ensemble_bias=True,
                    scaling_init='ones',
                )
                _init_first_adapter(
                    _get_first_ensemble_layer(self.backbone).r,  # type: ignore[code]
                    first_adapter_init,
                    first_adapter_sections,
                )

            else:
                raise ValueError(f'Unknown arch_type: {arch_type}')

        # >>> Output
        d_block = backbone['d_block']
        d_out = 1 if n_classes is None else n_classes
        self.output = (
            nn.Linear(d_block, d_out)
            if arch_type == 'plain'
            else NLinear(k, d_block, d_out)  # type: ignore[code]
        )

        # >>>
        self.arch_type = arch_type
        self.k = k

    def forward(
        self, x_num: None | Tensor = None, x_cat: None | Tensor = None
    ) -> Tensor:
        x = []
        if x_num is not None:
            x.append(x_num if self.num_module is None else self.num_module(x_num))
        if x_cat is None:
            assert self.cat_module is None
        else:
            assert self.cat_module is not None
            x.append(self.cat_module(x_cat).float())
        x = torch.column_stack([x_.flatten(1, -1) for x_ in x])

        if self.k is not None:
            x = x[:, None].expand(-1, self.k, -1)  # (B, D) -> (B, K, D)
            if self.minimal_ensemble_adapter is not None:
                x = self.minimal_ensemble_adapter(x)
        else:
            assert self.minimal_ensemble_adapter is None

        x = self.backbone(x)
        x = self.output(x)
        if self.k is None:
            # Adjust the output shape for plain networks to make them compatible
            # with the rest of the script (loss, metrics, predictions, ...).
            # (B, D_OUT) -> (B, 1, D_OUT)
            x = x[:, None]
        return x

CPU times: user 2.19 s, sys: 538 ms, total: 2.73 s
Wall time: 4.62 s


In [11]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = Model(
        n_num_features=79,
        cat_cardinalities=[],
        n_classes=None,
        bins=None,
        backbone={
            'type': 'MLP',
            'n_blocks': 3,
            'd_block': 512,
            'dropout': 0.1,
        },
        arch_type='tabm',
        k=1,
    ).to(device)
checkpoint = torch.load('/kaggle/input/tabm-model-full/latest0.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval() 

/tmp/ipykernel_23/1899230727.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('/kaggle/input/tabm-model-full/latest0.pth', map_location=device)


Model(
  (backbone): MLP(
    (blocks): ModuleList(
      (0-2): 3 x Sequential(
        (0): LinearEfficientEnsemble()
        (1): ReLU()
        (2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output): NLinear()
)

In [12]:
import pandas as pd
import polars as pl
import numpy as np
import os
import gc
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import KFold
#import xgboost as xgb
#from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor, log_evaluation
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import pickle

gc.enable()

path = '/kaggle/input/jane-street-real-time-market-data-forecasting/'
models_path = '/kaggle/input/js_lgb_20250112_04/other/default/1'
model_names = os.listdir(models_path)
models_list = []
for model_name in model_names:
    with open(f"{models_path}/{model_name}", "rb") as f:
        models_list.append(pickle.load(f))
lags_df : pl.DataFrame | None = None

# Replace this function with your inference code.
# You can return either a Pandas or Polars dataframe, though Polars is recommended.
# Each batch of predictions (except the very first) must be returned within 1 minute of the batch features being provided.
def predict_lgb(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame | pd.DataFrame:
    """Make a prediction."""
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)]+ [f"responder_{idx}_lag_1" for idx in range(9)]
    # All the responders from the previous day are passed in at time_id == 0. We save them in a global variable for access at every time_id.
    # Use them as extra features, if you like.
    global lags_df,models_list
    if lags is not None:
        lags_df = lags

    #test_df = test.drop(['row_id', 'date_id', 'time_id', 'symbol_id', 'is_scored', 'weight']).select(pl.all().shrink_dtype()).to_pandas()
    test_df = test.drop(['row_id', 'is_scored', 'weight'])
    test_df = test_df.join(lags_df, on=['date_id', 'time_id', 'symbol_id'], how='left').drop(['date_id', 'time_id', 'symbol_id']).select(pl.all().shrink_dtype())
    
    '''print(test_df['date_id'][0])
    if test_df['date_id'][0] != 0:
        raise ValueError('first date id not 0')'''
    test_df = test_df.to_numpy()

    '''if lags is not None:
        lags_df = lags.group_by(["date_id", "symbol_id"], maintain_order=True).last().clone() # pick up last record of previous date
        
        test_df = test_df.join(lags_df, on=["date_id", "symbol_id"],  how="left")
    else:
        test_df = test_df.with_columns(
            (pl.lit(None).alias(f'responder_{idx}_lag_1') for idx in range(9))
        )'''

    
    
    preds_list = []
    for model in models_list:
        preds_list.append(model.predict(test_df))

    preds_mean = np.mean(preds_list, axis=0)

    
    print(preds_mean)
    return preds_mean

In [13]:
import polars as pl
values=[1696, 1697, 1698]
main_dt=pl.read_parquet('/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id=9').fill_null(0).filter(pl.col('date_id').is_in(values))
main_dt = main_dt.rename({"responder_6": "responder_6_lag_1"})

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import math 
import numpy as np
import math
import os
import random
import warnings
from typing import Literal
import pandas as pd
import numpy as np
import polars as pl
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import torch
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from tqdm.std import tqdm
i=0
import torch
import torch.nn.functional as F
import math
from tqdm import tqdm
def retrain_model(model):
    """
    Retrain the model on rolling window of data.
    
    Parameters:
    - model (torch.nn.Module): Pretrained model
    
    Returns:
    - model (torch.nn.Module): The retrained model
    """
    global main_dt, values
    
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    
    # Create copy of old model for knowledge distillation
    old_model = type(model)(*model.init_args).to(device) if hasattr(model, 'init_args') else None
    if old_model is not None:
        old_model.load_state_dict(model.state_dict())
        old_model.eval()
    
    # Prepare data
    main_dt_X = main_dt[[f"feature_{idx:02d}" for idx in range(79)]]
    main_dt_y = main_dt['responder_6_lag_1']
    X = torch.FloatTensor(main_dt_X.to_numpy().copy()).to(device)
    y = torch.FloatTensor(main_dt_y.to_numpy().copy()).to(device)
    
    print('✅ Training data loaded.')
    print(f'Current dates in training: {values}')
    
    # Shuffle the data
    indices = torch.randperm(X.shape[0], device=device)
    X = X[indices]
    y = y[indices]
    
    amp_dtype = (
        torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16 if torch.cuda.is_available()
        else None
    )
    
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
        
    optimizer = torch.optim.AdamW(
        make_parameter_groups(model),
        lr=1e-5,
        weight_decay=1e-4
    )
    
    amp_enabled = False and amp_dtype is not None
    grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None
    
    @torch.autocast(device.type, enabled=amp_enabled, dtype=amp_dtype)
    def apply_model(x: torch.Tensor) -> torch.Tensor:
        return model(x, None).squeeze(-1).float()
    
    def loss_fn(y_pred: torch.Tensor, y_true: torch.Tensor, 
                old_pred: torch.Tensor = None, temp: float = 2.0) -> torch.Tensor:
        # MSE loss on actual targets
        mse_loss = F.mse_loss(y_pred.flatten(0, 1), y_true.repeat_interleave(y_pred.shape[1]))
        
        # Knowledge distillation loss if old model predictions available
        if old_pred is not None:
            kd_loss = F.mse_loss(
                F.softmax(y_pred / temp, dim=1),
                F.softmax(old_pred / temp, dim=1)
            ) * (temp ** 2)
            return 0.7 * mse_loss + 0.3 * kd_loss
        
        return mse_loss
    
    n_epochs = 10
    batch_size = 8000
    epoch_size = math.ceil(len(y) / batch_size)
    
    print(f"🔄 Starting training loop for {n_epochs} epochs...")
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0  # Initialize epoch loss
        batch_count = 0  # Track the number of batches
    
        for batch_idx in tqdm(
            torch.randperm(len(y), device=device).split(batch_size),
            desc=f'Epoch {epoch}',
            total=epoch_size,
        ):
            optimizer.zero_grad()
            pred = apply_model(X[batch_idx])
            old_pred = old_model(X[batch_idx], None).squeeze(-1) if old_model else None
    
            loss = loss_fn(pred, y[batch_idx], old_pred)
            epoch_loss += loss.item()  # Accumulate the loss value
            batch_count += 1  # Increment batch count
    
            if grad_scaler is None:
                loss.backward()
                optimizer.step()
            else:
                grad_scaler.scale(loss).backward()
                grad_scaler.step(optimizer)
                grad_scaler.update()
    
        # Calculate and print average loss for the epoch
        average_loss = epoch_loss / batch_count
        print(f'Epoch {epoch} - Average Loss: {average_loss:.4f}')


    # Update rolling window
    if len(values) >= 4:
        oldest_date = values[0]
        main_dt = main_dt.filter(main_dt["date_id"] != oldest_date)
        values.pop(0)  # Remove oldest date
        print(f'Removed oldest date {oldest_date} from training window')
    
    print("✅ Training complete.")
    print(f'Current dates in window: {values}')
    return model

In [15]:
%%time
import polars as pl
import pandas as pd
import numpy as np

# Load the pre-trained CatBoost models
temp=pl.DataFrame()
lags_= pl.DataFrame()
day=0
# Prediction function
def predict(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame | pd.DataFrame:
    """Generate predictions for responder_6 using the average of two models."""
    global temp, model, lags_, device,day, main_dt,values
    pred_lgb=predict_lgb(test,lags)
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)]
    yp = None
    # print(lags_.shape, temp.shape)
    # Retraining if lags and temp are available
    if lags is not None and temp.height!=0:
        print("lagstemp")
    # if temp.height!=0:
    #     print('temp')
    if lags is not None and temp.height!=0:
        print("OHHKAY")
        lags_=pl.concat([lags_,lags])
        day+=1
        if(day>=1):
            mdf_up = True
            batch_size = 512
            print(1)
            dfts=pl.concat([temp,lags_["responder_6_lag_1"].to_frame()],how='horizontal')
            print(f"dfts append:  {dfts.shape}")
            main_dt=pl.concat([main_dt,dfts],how="diagonal")
            unique_date_ids_list = dfts["date_id"].unique().to_list()
            values.extend(unique_date_ids_list)
            # print(unique_date_ids_list)
            print(f"main_dt append:  {main_dt.shape}")
            model=retrain_model(model)
            del unique_date_ids_list,dfts
            temp = pl.DataFrame()
            lags_=pl.DataFrame()
            day=0
        # print(model.state_dict())

        
        
        # print(model.state_dict())
          # Reset temp after retraining
    
    # Fill nulls in the DataFrame (replace with 3 or another constant)
    X = test[feature_cols].fill_null(0)
    xt=test[feature_cols+['date_id']].fill_null(0)
    # print(X.shape)
    temp=pl.concat([temp,xt])
    del xt
    # Convert to numpy array efficiently (after fill_null)
    
    
    
    # Memory-efficient batch processing
    batch_size = 512
    
    # Convert numpy array to tensor for prediction
    X_tensor = torch.tensor(X.to_numpy(), device=device).float()
    
    # Use Dataset and DataLoader for batching
    dataset = TensorDataset(X_tensor)
    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=device.type == 'cpu'
    )
    
    # Preallocate predictions list
    yp = []
    
    # Inference (without gradients)
    with torch.no_grad():
        model.eval()  # Set model to evaluation mode
        
        # Use mixed precision if using CUDA (GPU)
        if device.type == 'cuda':
            inference_context = torch.amp.autocast('cuda')
            with inference_context:
                for batch_X in data_loader:
                    batch_X = batch_X[0].to(device, non_blocking=True)
                    batch_pred = model(batch_X)
                    yp.append(batch_pred.detach().cpu().numpy())
                    torch.cuda.empty_cache()  # Clear GPU memory
        else:
            for batch_X in data_loader:
                batch_X = batch_X[0].to(device, non_blocking=True)
                batch_pred = model(batch_X)
                yp.append(batch_pred.detach().cpu().numpy())
    
    # Concatenate all predictions into a single array and flatten
    yp = np.concatenate(yp, axis=0).flatten()
    yp=(0.7*yp)+(0.3*pred_lgb)
    yp = np.nan_to_num(yp, nan=0.0)
    # Update temp for future retraining
    yp = np.clip(yp, -5, 5)

# Add assertion to check if all predictions are valid
    assert np.all((yp >= -5) & (yp <= 5)), "Some predictions are out of the valid range [-5, 5]"

    
    # Return results as a Polars DataFrame
    predictions = pl.DataFrame({
        'row_id': test['row_id'],  # Convert row_id column to numpy
        'responder_6': yp  # Add predictions as responder_6 column
    })
    
    # Ensure correct structure and types
    assert set(predictions.columns) == {'row_id', 'responder_6'}, f"Columns mismatch: {predictions.columns}"
    assert len(predictions) == len(test), f"Length mismatch: {len(predictions)} vs {len(test)}"
    print(predictions)
    return predictions



CPU times: user 260 µs, sys: 0 ns, total: 260 µs
Wall time: 265 µs


In [16]:
inference_server = js_server.JSInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
            '/kaggle/input/jane-street-real-time-market-data-forecasting/test.parquet',
            '/kaggle/input/jane-street-real-time-market-data-forecasting/lags.parquet',
        )
    )
    

[ 0.06073455  0.05106982  0.06064467  0.05676032  0.05079507  0.05635574
  0.01023787  0.03425415 -0.00037903  0.07216685  0.05062624  0.05476929
  0.05762082  0.06787792  0.02772713  0.02772713  0.0841009  -0.03121128
 -0.05179243  0.05238152 -0.10824836  0.06919263  0.00535841 -0.02983846
  0.02825169  0.04563197  0.05079507 -0.00196432  0.02603436  0.06406387
  0.06787792 -0.10873394  0.02406368  0.05974184  0.05212281  0.05861353
  0.05938     0.05650161  0.03028495]
shape: (39, 2)
┌────────┬─────────────┐
│ row_id ┆ responder_6 │
│ ---    ┆ ---         │
│ i64    ┆ f64         │
╞════════╪═════════════╡
│ 0      ┆ 0.018371    │
│ 1      ┆ 0.015472    │
│ 2      ┆ 0.018345    │
│ 3      ┆ 0.017179    │
│ 4      ┆ 0.01539     │
│ …      ┆ …           │
│ 34     ┆ 0.015788    │
│ 35     ┆ 0.017735    │
│ 36     ┆ 0.017965    │
│ 37     ┆ 0.017102    │
│ 38     ┆ 0.009237    │
└────────┴─────────────┘
